In [1]:
!pip install -q ultralytics

from google.colab import drive
drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive


In [5]:
%pip install -q roboflow
%pip install -q roboflow ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 12.3 MB/s eta 0:00:00


In [6]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY").strip()
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

In [7]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = rf.workspace(
    "safety-object-detection-app"
).project(
    "construction-site-safety-ccp7o"
)

version = project.version(1)
dataset = version.download("yolov11")

DATA_YAML = f"{dataset.location}/data.yaml"

print("Dataset folder:", dataset.location)
print(open(DATA_YAML).read())

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-Site-Safety-1 in yolov11:: 100%|██████████| 5206/5206 [00:00<00:00, 6628.30it/s]

Dataset folder: /content/Construction-Site-Safety-1
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 10
names: ['Hardhat', 'Machinery', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'Vehicle']

roboflow:
  workspace: safety-object-detection-app
  project: construction-site-safety-ccp7o
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/safety-object-detection-app/construction-site-safety-ccp7o/dataset/1


In [8]:
from pathlib import Path
import yaml

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("Dataset YAML:", DATA_YAML)
print("Train:", config.get("train"))
print("Validation:", config.get("val"))
print("Test:", config.get("test"))

dataset_dir = Path(DATA_YAML).parent
for split in ("train", "valid", "val", "test"):
    image_dir = dataset_dir / split / "images"
    if image_dir.exists():
        print(f"{split}: {len(list(image_dir.glob('*')))} images")

Dataset YAML: /content/Construction-Site-Safety-1/data.yaml
Train: ../train/images
Validation: ../valid/images
Test: ../test/images
train: 1819 images
valid: 518 images
test: 260 images


In [9]:
from pathlib import Path

DATA_YAML = str(Path(dataset.location) / "data.yaml")
print("Dataset config:", DATA_YAML)
print("Found:", Path(DATA_YAML).is_file())

Dataset config: /content/Construction-Site-Safety-1/data.yaml
Found: True


In [15]:
from pathlib import Path
import torch
from ultralytics import YOLO


model = YOLO("yolo11s.pt")
device = 0 if torch.cuda.is_available() else "cpu"

model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.001,
    workers=8,
    device=0,
    seed=0,
    pretrained=True,
    project="/content/drive/MyDrive/new",
    name="construction_yolo11s_v1",
    plots=True,
)

print("Training output:", model.trainer.save_dir)


Ultralytics 8.4.179 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=construction_yolo11s_v1, nbs=64, nms=None, 

In [16]:
from pathlib import Path

runs_dir = Path("/content/drive/MyDrive/new")
run_dir = runs_dir / "construction_yolo11s_v1"

print("Runs folder exists:", runs_dir.exists())
print("YOLO11 run exists:", run_dir.exists())

if runs_dir.exists():
    print("Folders in runs:")
    for item in runs_dir.iterdir():
        print(item.name)

print("Best checkpoint exists:", (run_dir / "weights" / "best.pt").exists())

Runs folder exists: True
YOLO11 run exists: True
Folders in runs:
construction_yolo11s_v1
Best checkpoint exists: True


In [18]:
best_model = YOLO(
    "/content/drive/MyDrive/new/construction_yolo11s_v1/weights/best.pt"
)

metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    batch=16,
    device=device,
    plots=True
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

Ultralytics 8.4.179 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,416,670 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 22.5±10.6 MB/s, size: 62.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/Construction-Site-Safety-1/test/labels... 260 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 260/260 682.9it/s 0.4s
val: New cache created: /content/Construction-Site-Safety-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.4it/s 7.2s
                   all        260       3733      0.889      0.737      0.805      0.567
               Hardhat        131        310      0.887       0.59      0.669      0.444
             Machinery        206        511      0.931      0.869   

In [26]:
from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [27]:
from pathlib import Path
import shutil

candidates = [
    Path("/content/drive/MyDrive/new/construction_yolo11s_v1"),
    Path("/content/drive/MyDrive/ppe_detection/runs/construction_yolo11s_v1"),
]

src = next(
    (p for p in candidates if (p / "weights" / "best.pt").is_file()),
    None
)

if src is None:
    print("Could not find best.pt. Check the Colab Files pane for the run folder's actual path.")
else:
    dst = Path("/content/gdrive/MyDrive/new/construction_yolo11s_v1")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Copied from:", src)
    print("Copied to:", dst)
    print("Checkpoint copied:", (dst / "weights" / "best.pt").is_file())

Copied from: /content/drive/MyDrive/new/construction_yolo11s_v1
Copied to: /content/gdrive/MyDrive/new/construction_yolo11s_v1
Checkpoint copied: True
